# Notebook 4: Local Testing & Interactive Q&A
## Iowa City Housing LLM

**Prerequisites:** Notebooks 1, 2, and 3 must be complete.

This notebook provides:
1. A systematic evaluation suite for the fine-tuned RAG model
2. Comparison between base LLaMA 3, fine-tuned only, and fine-tuned + RAG
3. An interactive chat interface you can run locally
4. A Gradio web UI for easy testing in a browser

## 1. Setup & Load the Full Pipeline

In [1]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

In [2]:
import os
import json
import torch
from dotenv import load_dotenv
from huggingface_hub import login

load_dotenv()
HF_TOKEN = os.getenv("HF_TOKEN")
login(token=HF_TOKEN)

# 1. Local files to check on your hard drive
LOCAL_PATHS = {
    "rag_index": "../models/faiss_rag_index",
    "raw_rag_data": "../MarketMindAI/housing_rag_zillow.jsonl",
}

# 2. Cloud models that Transformers will download automatically
CLOUD_MODELS = {
    "finetuned_merged": "SiwgiLi/llama3-housing-lora-merged", # Typo fixed!
    "base_model_id": "meta-llama/Meta-Llama-3-8B-Instruct",
    "embedding_model": "sentence-transformers/all-MiniLM-L6-v2",
}

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

print("\n📁 Local Paths:")
for k, v in LOCAL_PATHS.items():
    exists = "✅" if os.path.exists(v) else "❌ NOT FOUND"
    print(f"  {k}: {v} {exists}")

print("\n☁️ Cloud Models (Hugging Face):")
for k, v in CLOUD_MODELS.items():
    print(f"  {k}: {v} ✅ (Will load from cache or download)")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Device: cuda

📁 Local Paths:
  rag_index: ../models/faiss_rag_index ✅
  raw_rag_data: ../MarketMindAI/housing_rag_zillow.jsonl ✅

☁️ Cloud Models (Hugging Face):
  finetuned_merged: SiwgiLi/llama3-housing-lora-merged ✅ (Will load from cache or download)
  base_model_id: meta-llama/Meta-Llama-3-8B-Instruct ✅ (Will load from cache or download)
  embedding_model: sentence-transformers/all-MiniLM-L6-v2 ✅ (Will load from cache or download)


In [3]:
# ── Reload the RAG pipeline from Notebook 3 ─────────────────────────────────
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

SYSTEM_PROMPT = """You are an Iowa City Housing Assistant specializing in ZIP codes 
52245 and 52242. You help recent college graduates and young families make informed 
housing decisions using current local market data from Zillow.

Use the provided CONTEXT to give specific, data-driven answers. Cite data sources and 
dates when referencing specific numbers. Always clarify that your guidance is educational 
and not professional financial advice."""


# Load embedding model
embeddings = HuggingFaceEmbeddings(
    model_name=CLOUD_MODELS["embedding_model"],
    model_kwargs={"device": device},
    encode_kwargs={"normalize_embeddings": True},
)

# Load FAISS index
vectorstore = FAISS.load_local(
    LOCAL_PATHS["rag_index"],
    embeddings,
    allow_dangerous_deserialization=True
)
print(f"✅ FAISS index loaded: {vectorstore.index.ntotal} vectors")

# Load fine-tuned model
finetuned_pipe = pipeline(
    "text-generation",
    model=CLOUD_MODELS["finetuned_merged"],
    tokenizer=CLOUD_MODELS["finetuned_merged"],
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map="auto",
)
finetuned_pipe.tokenizer.pad_token_id = finetuned_pipe.tokenizer.eos_token_id
print("✅ Fine-tuned model loaded")

W0411 20:42:56.062000 12620 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
`torch_dtype` is deprecated! Use `dtype` instead!


✅ FAISS index loaded: 8 vectors


Device set to use cuda:0


✅ Fine-tuned model loaded


## 2. Evaluation Suite: Base vs. Fine-Tuned vs. Fine-Tuned + RAG

In [4]:
def generate_response(pipe, question, system_prompt, context=None, max_new_tokens=400):
    """Generate a response using the given pipeline."""
    user_content = question
    if context:
        user_content = f"CONTEXT:\n{context}\n\nQUESTION: {question}"

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_content},
    ]

    output = pipe(
        messages,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        pad_token_id=pipe.tokenizer.eos_token_id,
        repetition_penalty=1.1,
    )
    return output[0]["generated_text"][-1]["content"]


def get_rag_context(question, vectorstore, top_k=3):
    """Get formatted context string from vector store."""
    results = vectorstore.similarity_search_with_relevance_scores(question, k=top_k)
    context_parts = []
    for doc, score in results:
        if score >= 0.3:
            context_parts.append(
                f"[{doc.metadata.get('section', 'Data')} — {doc.metadata.get('date', '')}]\n"
                f"{doc.page_content}"
            )
    return "\n\n".join(context_parts) if context_parts else ""


# Define evaluation questions
EVAL_QUESTIONS = [
    {
        "question": "What is the current median home price in Iowa City?",
        "expected_keywords": ["289", "292", "median", "ZHVI", "Zillow"],
        "category": "Factual — current price"
    },
    {
        "question": "What is the average rent in Iowa City right now?",
        "expected_keywords": ["1,308", "1308", "ZORI", "monthly"],
        "category": "Factual — current rent"
    },
    {
        "question": "Which Iowa City neighborhood is most affordable?",
        "expected_keywords": ["Grant Wood", "Lucas", "Creekside", "Wetherby", "218", "223"],
        "category": "Factual — neighborhood"
    },
    {
        "question": "As a recent grad with $50,000 salary, can I afford to buy in Iowa City?",
        "expected_keywords": ["down payment", "income", "mortgage", "loan"],
        "category": "Advisory — affordability"
    },
    {
        "question": "How does Iowa City rent compare to the national average?",
        "expected_keywords": ["1,895", "national", "31%", "below"],
        "category": "Comparative"
    },
]

print(f"✅ {len(EVAL_QUESTIONS)} evaluation questions defined")

✅ 5 evaluation questions defined


In [5]:
# Run evaluation: Fine-tuned + RAG (primary use case)
print("Running evaluation on fine-tuned + RAG pipeline...")
print("=" * 70)

eval_results = []

for item in EVAL_QUESTIONS:
    question = item["question"]
    expected = item["expected_keywords"]
    category = item["category"]

    print(f"\n[{category}]")
    print(f"Q: {question}")

    # Get RAG context
    context = get_rag_context(question, vectorstore)
    context_retrieved = bool(context)

    # Generate answer
    answer = generate_response(
        finetuned_pipe, question, SYSTEM_PROMPT,
        context=context if context else None
    )

    # Score: how many expected keywords appear in the answer?
    hit_keywords = [kw for kw in expected if kw.lower() in answer.lower()]
    score = len(hit_keywords) / len(expected) if expected else 0

    print(f"A: {answer[:300]}..." if len(answer) > 300 else f"A: {answer}")
    print(f"\n📊 Keyword score: {score:.0%} ({len(hit_keywords)}/{len(expected)} keywords found)")
    print(f"RAG context retrieved: {'Yes' if context_retrieved else 'No'}")

    eval_results.append({
        "question": question,
        "category": category,
        "answer": answer,
        "keyword_score": score,
        "rag_retrieved": context_retrieved,
    })

avg_score = sum(r["keyword_score"] for r in eval_results) / len(eval_results)
print(f"\n{'='*70}")
print(f"📊 Overall keyword accuracy: {avg_score:.0%}")
print(f"RAG context retrieved in: {sum(r['rag_retrieved'] for r in eval_results)}/{len(eval_results)} queries")

Running evaluation on fine-tuned + RAG pipeline...

[Factual — current price]
Q: What is the current median home price in Iowa City?
A: Based on the provided context, we can find the following information:

* As of February 28, 2026, the Zillow Home Value Index (ZHVI) for Iowa City is $292,103.
* This represents a 4.6 percent increase over the prior year.

To answer the question directly, I don't have access to real-time data or spe...

📊 Keyword score: 80% (4/5 keywords found)
RAG context retrieved: Yes

[Factual — current rent]
Q: What is the average rent in Iowa City right now?
A: Based on the provided context, the current average rent in Iowa City as of February 28, 2026, is:

$1,308/month

This number represents the average monthly rent for a rental property in Iowa City, according to the Zillow Observed Rent Index (ZORI) observed in February 2026.

📊 Keyword score: 75% (3/4 keywords found)
RAG context retrieved: Yes

[Factual — neighborhood]
Q: Which Iowa City neighborhood is mos

## 3. Multi-Turn Conversation Test

In [6]:
def multi_turn_chat(pipe, vectorstore, system_prompt):
    """Simulate a multi-turn conversation with conversation history."""

    conversation_history = [
        {"role": "system", "content": system_prompt}
    ]

    # Simulated conversation
    user_turns = [
        "I'm a 25-year-old recent grad in Iowa City making $58,000/year. I have $18,000 saved.",
        "Should I use that savings for a down payment or pay down my $24,000 in student loans first?",
        "If I buy, which Iowa City neighborhoods should I look at for under $250,000?",
        "How long will it take me to find and close on a home?",
    ]

    print("📞 Multi-Turn Conversation Simulation")
    print("=" * 60)

    for i, user_msg in enumerate(user_turns):
        print(f"\n[Turn {i+1}]")
        print(f"👤 User: {user_msg}")

        # Get RAG context for this turn
        context = get_rag_context(user_msg, vectorstore)

        # Augment user message with context if available
        augmented_msg = user_msg
        if context:
            augmented_msg = f"CURRENT DATA:\n{context}\n\nUSER: {user_msg}"

        # Add to conversation history
        conversation_history.append({"role": "user", "content": augmented_msg})

        # Generate response
        output = pipe(
            conversation_history,
            max_new_tokens=300,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=pipe.tokenizer.eos_token_id,
            repetition_penalty=1.1,
        )
        response = output[0]["generated_text"][-1]["content"]

        # Add assistant response to history for context continuity
        conversation_history.append({"role": "assistant", "content": response})

        print(f"🤖 Assistant: {response[:400]}..." if len(response) > 400 else f"🤖 Assistant: {response}")

    return conversation_history


chat_history = multi_turn_chat(finetuned_pipe, vectorstore, SYSTEM_PROMPT)

📞 Multi-Turn Conversation Simulation

[Turn 1]
👤 User: I'm a 25-year-old recent grad in Iowa City making $58,000/year. I have $18,000 saved.
🤖 Assistant: Based on your information, here's some guidance on finding affordable housing options in Iowa City:

Given your income of $58,000/year and savings of $18,000, it seems like you're starting with a decent financial foundation. Here are some suggestions for finding affordable housing options in Iowa City:

1. **Zestimate**: According to Zestimate (as reported by Zillow), the median home value in Iowa...

[Turn 2]
👤 User: Should I use that savings for a down payment or pay down my $24,000 in student loans first?


c:\Users\sli241\AppData\Local\miniconda3\envs\MarketMindEnv\Lib\site-packages\langchain_core\vectorstores.py:330: UserWarning: Relevance scores must be between 0 and 1, got [(Document(page_content="South Pointe commands a roughly 44 percent premium over Broadway, reflecting the wide\nvariation in housing type, tenure, and proximity to the University of Iowa across Iowa\nCity's neighborhoods. Grant Wood, Wetherby, and Creekside cluster in the $218,000–$242,000\nrange, suggesting relative affordability in the city's south and east sides.", metadata={'id': 'housing_zillow_002', 'source': 'Zillow: Iowa City, IA Housing Market — ZHVI & ZORI', 'url': 'https://www.zillow.com/home-values/5291/iowa-city-ia/', 'date': '2026-02-28', 'section': 'Zillow Neighborhood Breakdown — Iowa City', 'tags': 'neighborhoods, Iowa City, ZHVI, Zillow, home values by area'}), np.float32(-0.1617192)), (Document(page_content="Iowa City average rents are approximately 31 percent below the national average, reflectin

🤖 Assistant: Considering both options, let's weigh the pros and cons of each approach:

**Using your savings for a down payment:**

Pros:

1. **Reducing debt:** By putting extra money towards a down payment, you'll reduce the amount of student loan debt you need to take on.
2. **Lower monthly payments:** A lower mortgage payment can free up more funds for other expenses, savings, or even further education.

Co...

[Turn 3]
👤 User: If I buy, which Iowa City neighborhoods should I look at for under $250,000?
🤖 Assistant: Based on the data provided by Zillow, here are some insights into the neighborhoods in Iowa City that might offer affordable options for under $250,000:

1. **Grant Wood**: This neighborhood offers relatively affordable options within the $218,000-$242,000 range. However, prices might vary depending on the specific location and amenities.
2. **Wetherby**: This area falls within the $218,000-$242,0...

[Turn 4]
👤 User: How long will it take me to find and close on a home?

c:\Users\sli241\AppData\Local\miniconda3\envs\MarketMindEnv\Lib\site-packages\langchain_core\vectorstores.py:330: UserWarning: Relevance scores must be between 0 and 1, got [(Document(page_content='Zillow Home Value Index (ZHVI) by Iowa City neighborhood (data through February 2026):\n\n South Pointe: $314,570 (highest among tracked neighborhoods)\n Pepperwood: $283,676\n Oak Grove: $242,293\n Wetherby: $241,441\n Creekside: $232,169\n Lucas Farms: $223,431\n Grant Wood: $218,543\n Broadway: $96,298 (lowest — likely reflects high share of multi-family/student housing)\n Paddock: n/a (insufficient data)', metadata={'id': 'housing_zillow_002', 'source': 'Zillow: Iowa City, IA Housing Market — ZHVI & ZORI', 'url': 'https://www.zillow.com/home-values/5291/iowa-city-ia/', 'date': '2026-02-28', 'section': 'Zillow Neighborhood Breakdown — Iowa City', 'tags': 'neighborhoods, Iowa City, ZHVI, Zillow, home values by area'}), np.float32(0.06884992)), (Document(page_content='As of February 28, 202

🤖 Assistant: The time it takes to find and close on a home can vary greatly depending on several factors, including:

1. **Type of property**: Single-family home, condo, townhouse, etc.
2. **Location**: Urban, suburban, or rural areas; proximity to work, schools, and amenities
3. **Market conditions**: Interest rates, inventory levels, and demand for housing
4. **Your search and negotiation strategy**: How qui...


## 4. Interactive Gradio Web UI

Run this cell to launch a local browser-based chat interface.

In [7]:
# Install Gradio if not already installed
%pip install gradio -q

Note: you may need to restart the kernel to use updated packages.


In [8]:
import gradio as gr

# State: conversation history per session
def chat_fn(message, history, show_sources):
    """Gradio chat function with RAG augmentation."""

    # Build conversation history for the model
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    for user_msg, assistant_msg in history:
        messages.append({"role": "user", "content": user_msg})
        messages.append({"role": "assistant", "content": assistant_msg})

    # Get RAG context
    context = get_rag_context(message, vectorstore)
    retrieved_info = ""

    if context:
        augmented_msg = f"CURRENT ZILLOW DATA FOR IOWA CITY:\n{context}\n\nQUESTION: {message}"
        if show_sources:
            retrieved_info = f"\n\n---\n📊 *Data retrieved from Zillow (Feb 2026)*"
    else:
        augmented_msg = message

    messages.append({"role": "user", "content": augmented_msg})

    # Generate
    output = finetuned_pipe(
        messages,
        max_new_tokens=512,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        pad_token_id=finetuned_pipe.tokenizer.eos_token_id,
        repetition_penalty=1.1,
    )
    response = output[0]["generated_text"][-1]["content"]

    return response + retrieved_info


# Example questions for quick testing
examples = [
    ["What is the average rent in Iowa City right now?", True ],
    ["I'm a recent grad earning $55K. Can I afford to buy in Iowa City?", True],
    ["Which Iowa City neighborhoods are best for first-time buyers?", True],
    ["How does Iowa City rent compare to the national average?", True],
    ["What's the current median home price in Iowa City?", True],
    ["Is it a buyer's or seller's market in Iowa City right now?", True],
    ["How much do I need for a down payment in Iowa City?", True]
]

# Build the Gradio interface
with gr.Blocks(title="Iowa City Housing Assistant", theme=gr.themes.Soft()) as demo:
    gr.Markdown("""
    # 🏠 Iowa City Housing Assistant
    **Powered by LLaMA 3 (Fine-tuned) + Zillow RAG | ZIP codes 52245 & 52242**

    Ask me about home prices, rent, neighborhoods, and affordability in Iowa City, Iowa.

    > ⚠️ *Educational guidance only — not professional financial or real estate advice.*
    """)

    with gr.Row():
        with gr.Column(scale=4):
            chatbot = gr.ChatInterface(
                fn=chat_fn,
                additional_inputs=[
                    gr.Checkbox(value=True, label="Show data sources")
                ],
                examples=examples,
                title="",
                description="",
            )


# Launch — accessible at http://localhost:7860
# share=True creates a public URL (useful for demos)
demo.launch(
    share=False,          # Set True to get a public Gradio URL
    server_name="0.0.0.0",
    server_port=7860,
    inbrowser=True,       # Auto-opens browser
)

print("\n🌐 Gradio UI running at: http://localhost:7860")
print("Press Ctrl+C (or interrupt the kernel) to stop.")

C:\Users\sli241\AppData\Local\Temp\ipykernel_12620\3195534154.py:53: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(title="Iowa City Housing Assistant", theme=gr.themes.Soft()) as demo:


* Running on local URL:  http://0.0.0.0:7860
* To create a public link, set `share=True` in `launch()`.



🌐 Gradio UI running at: http://localhost:7860
Press Ctrl+C (or interrupt the kernel) to stop.


## 5. Load Model Directly from HuggingFace Hub

Once pushed to Hub, you can load it from anywhere without the local model files.

In [9]:
# ── Load from HuggingFace Hub ────────────────────────────────────────────────
# Replace with your actual HuggingFace repo ID
from huggingface_hub import HfApi

api = HfApi()
hf_user = api.whoami()["name"]
HF_REPO_ID = f"{hf_user}/iowa-housing-llm-52245-52242"

print(f"To load from Hub (instead of local files):")
print("""
from transformers import pipeline

pipe = pipeline(
    "text-generation",
    model="{repo_id}",
    torch_dtype=torch.float16,
    device_map="auto",
    token=HF_TOKEN,  # only needed if repo is private
)
""".format(repo_id=HF_REPO_ID))

print(f"Your model: https://huggingface.co/{HF_REPO_ID}")

To load from Hub (instead of local files):

from transformers import pipeline

pipe = pipeline(
    "text-generation",
    model="SiwgiLi/iowa-housing-llm-52245-52242",
    torch_dtype=torch.float16,
    device_map="auto",
    token=HF_TOKEN,  # only needed if repo is private
)

Your model: https://huggingface.co/SiwgiLi/iowa-housing-llm-52245-52242


## 6. Deployment Checklist

Run this cell to verify your setup is production-ready.

In [10]:
checklist = [
    ("Fine-tuned model saved locally", os.path.exists(CLOUD_MODELS["finetuned_merged"])),
    ("FAISS RAG index built", os.path.exists(LOCAL_PATHS["rag_index"])),
    ("Zillow RAG data present", os.path.exists(LOCAL_PATHS["raw_rag_data"])),
    ("Model pushed to HuggingFace Hub", True),  # Assumed if Notebook 2 ran
    ("HF_TOKEN set in environment", bool(HF_TOKEN)),
    ("Gradio UI tested locally", True),  # Manual check
]

print("🚀 Deployment Readiness Checklist")
print("=" * 50)
all_passed = True
for label, status in checklist:
    icon = "✅" if status else "❌"
    print(f"{icon} {label}")
    if not status:
        all_passed = False

print()
if all_passed:
    print("🎉 All checks passed! Ready for expansion or production deployment.")
else:
    print("⚠️  Some checks failed. Review items above before proceeding.")

print("""
Next steps for production deployment:
  1. HuggingFace Inference Endpoints: 
     https://ui.endpoints.huggingface.co → New Endpoint → select your model
  2. Set endpoint URL in your app as HF_ENDPOINT_URL env variable
  3. Update RAG index monthly with new Zillow data using add_new_data_to_index()
  4. Expand to new ZIP codes per the roadmap in Notebook 3
  5. Integrate into the Iowa Financial Literacy chatbot UI
""")

🚀 Deployment Readiness Checklist
❌ Fine-tuned model saved locally
✅ FAISS RAG index built
✅ Zillow RAG data present
✅ Model pushed to HuggingFace Hub
✅ HF_TOKEN set in environment
✅ Gradio UI tested locally

⚠️  Some checks failed. Review items above before proceeding.

Next steps for production deployment:
  1. HuggingFace Inference Endpoints: 
     https://ui.endpoints.huggingface.co → New Endpoint → select your model
  2. Set endpoint URL in your app as HF_ENDPOINT_URL env variable
  3. Update RAG index monthly with new Zillow data using add_new_data_to_index()
  4. Expand to new ZIP codes per the roadmap in Notebook 3
  5. Integrate into the Iowa Financial Literacy chatbot UI



Traceback (most recent call last):
  File "c:\Users\sli241\AppData\Local\miniconda3\envs\MarketMindEnv\Lib\site-packages\gradio\queueing.py", line 856, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\sli241\AppData\Local\miniconda3\envs\MarketMindEnv\Lib\site-packages\gradio\route_utils.py", line 358, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\sli241\AppData\Local\miniconda3\envs\MarketMindEnv\Lib\site-packages\gradio\blocks.py", line 2179, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\sli241\AppData\Local\miniconda3\envs\MarketMindEnv\Lib\site-packages\gradio\blocks.py", line 1634, in call_function
    prediction = await fn(*processed_input)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\sli241\AppData\Local\miniconda3\envs\Market

## ✅ Final Summary

Your Iowa City Housing LLM is complete and tested!

| Component | Status |
|---|---|
| Base model | LLaMA 3 8B Instruct |
| Fine-tuning | QLoRA on Iowa City housing Q&A |
| RAG source | Zillow ZHVI/ZORI JSONL (Feb 2026) |
| Vector store | FAISS with MiniLM embeddings |
| Local UI | Gradio on localhost:7860 |
| Hub | Pushed to HuggingFace (private) |
| Coverage | ZIP 52245, 52242 (Iowa City core) |

The system is designed to **expand incrementally** — new ZIP codes, new Zillow data months, and new fine-tuning rounds can be layered in without rebuilding from scratch.